## **Step 1 — Load R2-5 data**

In [35]:
from pathlib import Path
import pandas as pd
import numpy as np

REPLICATION = "R2"

VALID_REPLICATIONS = {
    "R1", "R2", "R3", "R4", "R5"
}

assert REPLICATION in VALID_REPLICATIONS

release_dir = (
    Path("../data/robustness/releases")
    / REPLICATION
)

assert release_dir.is_dir(), (
    f"Missing replication directory: {release_dir.resolve()}"
)

print("Replication:", REPLICATION)
print("Data directory:", release_dir.resolve())

Replication: R2
Data directory: <PROJECT_ROOT>\data\robustness\releases\R2


In [36]:
features_df = pd.read_csv(
    release_dir / "account_month_features.csv"
)

labels_df = pd.read_csv(
    release_dir / "evaluation_labels.csv"
)

print("Replication:", REPLICATION)
print("Features shape:", features_df.shape)
print("Labels shape:", labels_df.shape)

display(labels_df.head())

Replication: R2
Features shape: (6000, 22)
Labels shape: (6000, 4)


,account_id,month,is_injected_anomaly,scenario
0,ACC00001,1,0,normal
1,ACC00001,2,0,normal
2,ACC00001,3,0,normal
3,ACC00001,4,1,S1_transaction_count_spike
4,ACC00001,5,0,normal


## **Validate R2-5 labels**

In [37]:
print("Features columns:")
print(features_df.columns.tolist())

print("\nLabels columns:")
print(labels_df.columns.tolist())

print("\nFirst five labels:")
display(labels_df.head())

Features columns:
['account_id', 'month', 'txn_count', 'unique_counterparties', 'inflow_sum', 'outflow_sum', 'inflow_avg', 'outflow_avg', 'net_flow', 'inflow_outflow_ratio', 'zero_outflow_flag', 'flow_concentration', 'txn_count_change_vs_prev', 'txn_count_robust_z_past', 'unique_counterparties_change_vs_prev', 'unique_counterparties_robust_z_past', 'inflow_sum_change_vs_prev', 'inflow_sum_robust_z_past', 'outflow_sum_change_vs_prev', 'outflow_sum_robust_z_past', 'net_flow_change_vs_prev', 'net_flow_robust_z_past']

Labels columns:
['account_id', 'month', 'is_injected_anomaly', 'scenario']

First five labels:


,account_id,month,is_injected_anomaly,scenario
0,ACC00001,1,0,normal
1,ACC00001,2,0,normal
2,ACC00001,3,0,normal
3,ACC00001,4,1,S1_transaction_count_spike
4,ACC00001,5,0,normal


## **Step 3 — Validate R2-5 and merge features with labels**

In [63]:
# Check dataset sizes
assert len(features_df) == 6000
assert len(labels_df) == 6000

# Check unique account-month keys
assert not features_df.duplicated(
    ["account_id", "month"]
).any()

assert not labels_df.duplicated(
    ["account_id", "month"]
).any()

# Check anomaly distribution
anomalies_by_month = (
    labels_df.groupby("month")["is_injected_anomaly"]
    .sum()
    .reindex(range(1, 7), fill_value=0)
)

print("Anomalies by month:")
print(anomalies_by_month)

assert anomalies_by_month.tolist() == [0, 0, 0, 50, 50, 50]

# Merge features and labels
df = features_df.merge(
    labels_df,
    on=["account_id", "month"],
    how="inner",
    validate="one_to_one"
)

assert len(df) == 6000

print("\nMerged dataset shape:", df.shape)
print(REPLICATION, " validation passed!")

Anomalies by month:
month
1     0
2     0
3     0
4    50
5    50
6    50
Name: is_injected_anomaly, dtype: int64

Merged dataset shape: (6000, 24)
R2  validation passed!


# **Step 4 — Chronological Train/Validation/Test Split**

In [39]:
# Chronological split
train_df = df[df["month"].isin([1, 2, 3])].copy()
val_df   = df[df["month"] == 4].copy()
test_df  = df[df["month"].isin([5, 6])].copy()

# Verify split sizes
assert len(train_df) == 3000
assert len(val_df) == 1000
assert len(test_df) == 2000

# Verify anomaly counts
assert train_df["is_injected_anomaly"].sum() == 0
assert val_df["is_injected_anomaly"].sum() == 50
assert test_df["is_injected_anomaly"].sum() == 100

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

print("\nAnomaly counts:")
print("Train:", train_df["is_injected_anomaly"].sum())
print("Validation:", val_df["is_injected_anomaly"].sum())
print("Test:", test_df["is_injected_anomaly"].sum())

Train: (3000, 24)
Validation: (1000, 24)
Test: (2000, 24)

Anomaly counts:
Train: 0
Validation: 50
Test: 100


# **Step 5 — Feature Selection**

**Step 5.1 — Define the four feature groups**

In [40]:
behavioral_features = [
    "txn_count",
    "unique_counterparties",
    "txn_count_change_vs_prev",
    "unique_counterparties_change_vs_prev"
]

fund_flow_core_features = [
    "inflow_sum",
    "outflow_sum",
    "inflow_avg",
    "outflow_avg",
    "net_flow",
    "inflow_outflow_ratio",
    "flow_concentration"
]

combined_core_features = (
    behavioral_features + fund_flow_core_features
)

fund_flow_extended_features = [
    "inflow_sum",
    "outflow_sum",
    "inflow_avg",
    "outflow_avg",
    "net_flow",
    "inflow_outflow_ratio",
    "zero_outflow_flag",
    "flow_concentration",
    "inflow_sum_change_vs_prev",
    "outflow_sum_change_vs_prev",
    "net_flow_change_vs_prev"
]

feature_groups = {
    "Behavioral": behavioral_features,
    "Fund Flow Core": fund_flow_core_features,
    "Combined Core": combined_core_features,
    "Fund Flow Extended": fund_flow_extended_features
}

**Step 5.2 — Validate feature availability**

In [42]:
for model_name, columns in feature_groups.items():

    missing = set(columns) - set(features_df.columns)

    assert not missing, (
        f"{model_name} has missing features: {missing}"
    )

    assert len(columns) == len(set(columns)), (
        f"{model_name} contains duplicate features"
    )

print("All feature groups validated!")

All feature groups validated!


**Step 5.3 — Prepare labels separately**

In [43]:
train_features = features_df[
    features_df["month"].isin([1, 2, 3])
].copy()

val_features = features_df[
    features_df["month"] == 4
].copy()

test_features = features_df[
    features_df["month"].isin([5, 6])
].copy()

train_labels = labels_df[
    labels_df["month"].isin([1, 2, 3])
].copy()

val_labels = labels_df[
    labels_df["month"] == 4
].copy()

test_labels = labels_df[
    labels_df["month"].isin([5, 6])
].copy()

# Explicitly align labels to feature rows by account-month keys
def aligned_targets(feature_rows, label_rows):
    aligned = feature_rows[
        ["account_id", "month"]
    ].merge(
        label_rows[
            ["account_id", "month", "is_injected_anomaly"]
        ],
        on=["account_id", "month"],
        how="left",
        validate="one_to_one"
    )

    assert aligned["is_injected_anomaly"].notna().all()

    return aligned["is_injected_anomaly"].to_numpy(dtype=int)

y_train = aligned_targets(train_features, train_labels)
y_val = aligned_targets(val_features, val_labels)
y_test = aligned_targets(test_features, test_labels)

print("Train:", len(y_train), "anomalies:", y_train.sum())
print("Validation:", len(y_val), "anomalies:", y_val.sum())
print("Test:", len(y_test), "anomalies:", y_test.sum())

Train: 3000 anomalies: 0
Validation: 1000 anomalies: 50
Test: 2000 anomalies: 100


# **Step 6 — Median Imputation**

**Step 6.1 — Inspect missing values**

In [44]:
for name, columns in feature_groups.items():
    print(f"\n{name}")

    print(
        train_features[columns]
        .isna()
        .sum()
        .loc[lambda x: x > 0]
    )


Behavioral
txn_count_change_vs_prev                1000
unique_counterparties_change_vs_prev    1000
dtype: int64

Fund Flow Core
inflow_avg              14
outflow_avg             16
inflow_outflow_ratio    16
flow_concentration       1
dtype: int64

Combined Core
txn_count_change_vs_prev                1000
unique_counterparties_change_vs_prev    1000
inflow_avg                                14
outflow_avg                               16
inflow_outflow_ratio                      16
flow_concentration                         1
dtype: int64

Fund Flow Extended
inflow_avg                      14
outflow_avg                     16
inflow_outflow_ratio            16
flow_concentration               1
inflow_sum_change_vs_prev     1008
outflow_sum_change_vs_prev    1012
net_flow_change_vs_prev       1000
dtype: int64


**Step 6.2 — Create an imputation function**

In [45]:
def prepare_model_features(
    train_df,
    val_df,
    test_df,
    columns
):
    # Select only the model's features
    X_train = train_df[columns].copy()
    X_val = val_df[columns].copy()
    X_test = test_df[columns].copy()

    # Calculate medians using TRAIN ONLY
    train_medians = X_train.median()

    # Guard against features with no observed training values
    if train_medians.isna().any():
        missing = train_medians[
            train_medians.isna()
        ].index.tolist()

        raise ValueError(
            f"All-missing training features: {missing}"
        )

    # Apply frozen training medians
    X_train = X_train.fillna(train_medians)
    X_val = X_val.fillna(train_medians)
    X_test = X_test.fillna(train_medians)

    # Validate preprocessing
    assert not X_train.isna().any().any()
    assert not X_val.isna().any().any()
    assert not X_test.isna().any().any()

    return X_train, X_val, X_test, train_medians

**Step 6.3 — Prepare all four models**

In [46]:
prepared_data = {}

for model_name, columns in feature_groups.items():

    X_train, X_val, X_test, medians = (
        prepare_model_features(
            train_features,
            val_features,
            test_features,
            columns
        )
    )

    prepared_data[model_name] = {
        "X_train": X_train,
        "X_val": X_val,
        "X_test": X_test,
        "medians": medians
    }

    print(
        f"{model_name}: "
        f"Train={X_train.shape}, "
        f"Validation={X_val.shape}, "
        f"Test={X_test.shape}"
    )

Behavioral: Train=(3000, 4), Validation=(1000, 4), Test=(2000, 4)
Fund Flow Core: Train=(3000, 7), Validation=(1000, 7), Test=(2000, 7)
Combined Core: Train=(3000, 11), Validation=(1000, 11), Test=(2000, 11)
Fund Flow Extended: Train=(3000, 11), Validation=(1000, 11), Test=(2000, 11)


# **Step 7 — Train the four Isolation Forest models**

In [47]:
IF_SETTINGS = {
    "random_state": 42,
    "n_estimators": 200,
    "max_samples": "auto",
    "contamination": "auto"
}

**Step 7.1 — Train and calculate anomaly scores**

In [48]:
from sklearn.ensemble import IsolationForest

models = {}
scores = {}

for model_name, data in prepared_data.items():

    model = IsolationForest(**IF_SETTINGS)

    model.fit(data["X_train"])

    models[model_name] = model

    scores[model_name] = {
        "train": -model.score_samples(data["X_train"]),
        "val": -model.score_samples(data["X_val"]),
        "test": -model.score_samples(data["X_test"])
    }

    print(
        f"{model_name}: "
        f"trained successfully | "
        f"Validation scores={len(scores[model_name]['val'])} | "
        f"Test scores={len(scores[model_name]['test'])}"
    )

Behavioral: trained successfully | Validation scores=1000 | Test scores=2000
Fund Flow Core: trained successfully | Validation scores=1000 | Test scores=2000
Combined Core: trained successfully | Validation scores=1000 | Test scores=2000
Fund Flow Extended: trained successfully | Validation scores=1000 | Test scores=2000


**Step 7.2 — Verify score distributions**

In [49]:
for model_name, model_scores in scores.items():

    print(f"\n{model_name}")

    for split in ["train", "val", "test"]:
        values = model_scores[split]

        print(
            f"{split}: "
            f"min={values.min():.4f}, "
            f"mean={values.mean():.4f}, "
            f"max={values.max():.4f}"
        )


Behavioral
train: min=0.3490, mean=0.4329, max=0.7787
val: min=0.3504, mean=0.4549, max=0.7710
test: min=0.3490, mean=0.4591, max=0.7732

Fund Flow Core
train: min=0.3563, mean=0.4357, max=0.7355
val: min=0.3580, mean=0.4378, max=0.7332
test: min=0.3572, mean=0.4397, max=0.7363

Combined Core
train: min=0.3492, mean=0.4257, max=0.6797
val: min=0.3525, mean=0.4341, max=0.6619
test: min=0.3513, mean=0.4368, max=0.6633

Fund Flow Extended
train: min=0.3373, mean=0.4025, max=0.6990
val: min=0.3398, mean=0.4121, max=0.7112
test: min=0.3412, mean=0.4147, max=0.7357


# **Step 8 — Validation threshold calibration**

**Step 8.1 — Define the threshold selection function**

In [50]:
from sklearn.metrics import precision_score, recall_score, f1_score
import numpy as np

def select_best_threshold(y_true, anomaly_scores):

    thresholds = np.append(
        np.unique(anomaly_scores),
        np.nextafter(np.max(anomaly_scores), np.inf)
    )
    results = []

    for threshold in thresholds:

        y_pred = (
            anomaly_scores >= threshold
        ).astype(int)

        precision = precision_score(
            y_true, y_pred, zero_division=0
        )

        recall = recall_score(
            y_true, y_pred, zero_division=0
        )

        f1 = f1_score(
            y_true, y_pred, zero_division=0
        )

        fp = int(((y_true == 0) & (y_pred == 1)).sum())

        results.append({
            "threshold": float(threshold),
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "fp": fp
        })

    best = sorted(
        results,
        key=lambda r: (
            -r["f1"],
            r["fp"],
            -r["threshold"]
        )
    )[0]

    return best

**Step 8.2 — Calibrate all four models**

In [51]:
threshold_results = {}

for model_name in feature_groups:

    result = select_best_threshold(
        y_val,
        scores[model_name]["val"]
    )

    threshold_results[model_name] = result

    print(f"\n{model_name}")
    print(f"Threshold: {result['threshold']:.6f}")
    print(f"Precision: {result['precision']:.4f}")
    print(f"Recall:    {result['recall']:.4f}")
    print(f"F1:        {result['f1']:.4f}")
    print(f"FP:        {result['fp']}")


Behavioral
Threshold: 0.530539
Precision: 0.1419
Recall:    0.4400
F1:        0.2146
FP:        133

Fund Flow Core
Threshold: 0.621709
Precision: 0.4091
Recall:    0.1800
F1:        0.2500
FP:        13

Combined Core
Threshold: 0.539659
Precision: 0.2308
Recall:    0.3600
F1:        0.2812
FP:        60

Fund Flow Extended
Threshold: 0.574432
Precision: 0.4400
Recall:    0.2200
F1:        0.2933
FP:        14


## **Step 9 — Evaluate R2-5 on the test dataset**

**Step 9.1 — Calculate test metrics**

In [52]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    confusion_matrix
)

test_results = {}

for model_name in feature_groups:

    threshold = threshold_results[model_name]["threshold"]
    test_scores = scores[model_name]["test"]

    y_pred = (test_scores >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_test, y_pred, labels=[0, 1]
    ).ravel()

    test_results[model_name] = {
        "threshold": threshold,
        "TP": int(tp),
        "FP": int(fp),
        "FN": int(fn),
        "TN": int(tn),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "AP": average_precision_score(y_test, test_scores),
        "alerts": int(y_pred.sum())
    }

results_df = pd.DataFrame.from_dict(
    test_results, orient="index"
)

display(results_df.round(4))

,threshold,TP,FP,FN,TN,precision,recall,f1,AP,alerts
Behavioral,0.5305,38,269,62,1631,0.1238,0.38,0.1867,0.1182,307
Fund Flow Core,0.6217,18,28,82,1872,0.3913,0.18,0.2466,0.1914,46
Combined Core,0.5397,36,133,64,1767,0.2130,0.36,0.2677,0.1927,169
Fund Flow Extended,0.5744,26,45,74,1855,0.3662,0.26,0.3041,0.2580,71


**Step 9.2 — Precision@100 and Recall@100**

In [53]:
K = 100

for model_name in feature_groups:

    test_scores = scores[model_name]["test"]

    # Stable ranking for reproducibility
    top_indices = np.argsort(
        -test_scores, kind="stable"
    )[:K]

    top_true_positives = int(y_test[top_indices].sum())

    precision_at_100 = top_true_positives / K
    recall_at_100 = top_true_positives / int(y_test.sum())

    test_results[model_name]["Precision@100"] = precision_at_100
    test_results[model_name]["Recall@100"] = recall_at_100

results_df = pd.DataFrame.from_dict(
    test_results, orient="index"
)

display(results_df.round(4))

,threshold,TP,FP,FN,TN,precision,recall,f1,AP,alerts,Precision@100,Recall@100
Behavioral,0.5305,38,269,62,1631,0.1238,0.38,0.1867,0.1182,307,0.23,0.23
Fund Flow Core,0.6217,18,28,82,1872,0.3913,0.18,0.2466,0.1914,46,0.28,0.28
Combined Core,0.5397,36,133,64,1767,0.2130,0.36,0.2677,0.1927,169,0.24,0.24
Fund Flow Extended,0.5744,26,45,74,1855,0.3662,0.26,0.3041,0.2580,71,0.28,0.28


# **Step 10 — Prepare R2-5 Fusion**

In [54]:
fusion_sources = ["Behavioral", "Fund Flow Core"]

standardized_scores = {}

for model_name in fusion_sources:

    train_scores = scores[model_name]["train"]

    train_mean = np.mean(train_scores)
    train_std = np.std(train_scores, ddof=0)

    assert train_std > 0

    standardized_scores[model_name] = {}

    for split in ["train", "val", "test"]:

        standardized_scores[model_name][split] = (
            scores[model_name][split] - train_mean
        ) / train_std

    print(
        f"{model_name}: "
        f"Train mean={train_mean:.6f}, "
        f"Train std={train_std:.6f}"
    )

Behavioral: Train mean=0.432949, Train std=0.071363
Fund Flow Core: Train mean=0.435662, Train std=0.065680


**Step 10.2 — Calculate Mean and Max Fusion scores**

In [55]:
fusion_scores = {
    "Mean Fusion": {},
    "Max Fusion": {}
}

for split in ["val", "test"]:

    behavioral = standardized_scores["Behavioral"][split]
    fund_flow = standardized_scores["Fund Flow Core"][split]

    fusion_scores["Mean Fusion"][split] = (
        behavioral + fund_flow
    ) / 2

    fusion_scores["Max Fusion"][split] = np.maximum(
        behavioral, fund_flow
    )

for fusion_name in fusion_scores:

    print(f"\n{fusion_name}")

    for split in ["val", "test"]:
        values = fusion_scores[fusion_name][split]

        print(
            f"{split}: "
            f"min={values.min():.4f}, "
            f"mean={values.mean():.4f}, "
            f"max={values.max():.4f}"
        )


Mean Fusion
val: min=-1.1267, mean=0.1694, max=3.6780
test: min=-1.1334, mean=0.2139, max=3.4499

Max Fusion
val: min=-1.1141, mean=0.7078, max=4.7375
test: min=-1.0724, mean=0.7630, max=4.7676


# **Step 11 — Calibrate Fusion thresholds**

In [56]:
fusion_threshold_results = {}

for fusion_name in ["Mean Fusion", "Max Fusion"]:

    result = select_best_threshold(
        y_val,
        fusion_scores[fusion_name]["val"]
    )

    fusion_threshold_results[fusion_name] = result

    print(f"\n{fusion_name}")
    print(f"Threshold: {result['threshold']:.6f}")
    print(f"Precision: {result['precision']:.4f}")
    print(f"Recall:    {result['recall']:.4f}")
    print(f"F1:        {result['f1']:.4f}")
    print(f"FP:        {result['fp']}")


Mean Fusion
Threshold: 1.823392
Precision: 0.2407
Recall:    0.2600
F1:        0.2500
FP:        41

Max Fusion
Threshold: 2.430840
Precision: 0.2381
Recall:    0.4000
F1:        0.2985
FP:        64


## **Step 12 — Evaluate Union, Mean and Max Fusion**

In [57]:
from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score
)

fusion_test_results = {}

for fusion_name in ["Union", "Mean Fusion", "Max Fusion"]:

    if fusion_name == "Union":

        behavioral_pred = (
            scores["Behavioral"]["test"] >=
            threshold_results["Behavioral"]["threshold"]
        )

        fund_flow_pred = (
            scores["Fund Flow Core"]["test"] >=
            threshold_results["Fund Flow Core"]["threshold"]
        )

        y_pred = (
            behavioral_pred | fund_flow_pred
        ).astype(int)

        threshold = None
        ap = np.nan

    else:

        threshold = fusion_threshold_results[
            fusion_name
        ]["threshold"]

        test_scores = fusion_scores[fusion_name]["test"]

        y_pred = (test_scores >= threshold).astype(int)

        ap = average_precision_score(
            y_test, test_scores
        )

    tn, fp, fn, tp = confusion_matrix(
        y_test, y_pred, labels=[0, 1]
    ).ravel()

    fusion_test_results[fusion_name] = {
        "threshold": threshold,
        "TP": int(tp),
        "FP": int(fp),
        "FN": int(fn),
        "TN": int(tn),
        "precision": precision_score(
            y_test, y_pred, zero_division=0
        ),
        "recall": recall_score(
            y_test, y_pred, zero_division=0
        ),
        "f1": f1_score(
            y_test, y_pred, zero_division=0
        ),
        "AP": ap,
        "alerts": int(y_pred.sum())
    }

fusion_results_df = pd.DataFrame.from_dict(
    fusion_test_results,
    orient="index"
)

display(fusion_results_df.round(4))

,threshold,TP,FP,FN,TN,precision,recall,f1,AP,alerts
Union,NaN,49,289,51,1611,0.1450,0.49,0.2237,NaN,338
Mean Fusion,1.8234,26,88,74,1812,0.2281,0.26,0.2430,0.1814,114
Max Fusion,2.4308,45,141,55,1759,0.2419,0.45,0.3147,0.1658,186


## **Step 13 — Compare Combined Core and Max Fusion**

In [58]:
comparison = pd.DataFrame({
    "Combined Core": test_results["Combined Core"],
    "Max Fusion": fusion_test_results["Max Fusion"]
}).T

display(
    comparison[
        ["TP", "FP", "precision", "recall",
         "f1", "AP", "alerts"]
    ].round(4)
)

,TP,FP,precision,recall,f1,AP,alerts
Combined Core,36.0,133.0,0.2130,0.36,0.2677,0.1927,169.0
Max Fusion,45.0,141.0,0.2419,0.45,0.3147,0.1658,186.0


## **Step 14 — Precision@100 and Recall@100 for Fusion**

In [60]:
K = 100

for fusion_name in ["Mean Fusion", "Max Fusion"]:

    test_scores = fusion_scores[fusion_name]["test"]

    top_indices = np.argsort(
        -test_scores, kind="stable"
    )[:K]

    tp_at_100 = int(y_test[top_indices].sum())

    precision_at_100 = tp_at_100 / K
    recall_at_100 = tp_at_100 / int(y_test.sum())

    fusion_test_results[fusion_name]["Precision@100"] = (
        precision_at_100
    )

    fusion_test_results[fusion_name]["Recall@100"] = (
        recall_at_100
    )

    print(f"\n{fusion_name}")
    print(f"TP@100: {tp_at_100}")
    print(f"Precision@100: {precision_at_100:.4f}")
    print(f"Recall@100: {recall_at_100:.4f}")


Mean Fusion
TP@100: 26
Precision@100: 0.2600
Recall@100: 0.2600

Max Fusion
TP@100: 30
Precision@100: 0.3000
Recall@100: 0.3000


## **Step 15 — Save the complete R2-5 results**

In [61]:
replication_results = pd.concat([
    pd.DataFrame.from_dict(
        test_results, orient="index"
    ),
    pd.DataFrame.from_dict(
        fusion_test_results, orient="index"
    )
])

replication_results.index.name = "model"

replication_results.insert(
    0, "replication", REPLICATION
)

output_dir = Path("../results/robustness")
output_dir.mkdir(parents=True, exist_ok=True)

output_file = (
    output_dir / f"{REPLICATION}_results.csv"
)

replication_results.to_csv(output_file)

print("Saved:", output_file.resolve())

display(replication_results.round(4))

Saved: <PROJECT_ROOT>\results\robustness\R2_results.csv


,replication,threshold,TP,FP,FN,TN,precision,recall,f1,AP,alerts,Precision@100,Recall@100
model,,,,,,,,,,,,,
Behavioral,R2,0.5305,38,269,62,1631,0.1238,0.38,0.1867,0.1182,307,0.23,0.23
Fund Flow Core,R2,0.6217,18,28,82,1872,0.3913,0.18,0.2466,0.1914,46,0.28,0.28
Combined Core,R2,0.5397,36,133,64,1767,0.2130,0.36,0.2677,0.1927,169,0.24,0.24
Fund Flow Extended,R2,0.5744,26,45,74,1855,0.3662,0.26,0.3041,0.2580,71,0.28,0.28
Union,R2,NaN,49,289,51,1611,0.1450,0.49,0.2237,NaN,338,NaN,NaN
Mean Fusion,R2,1.8234,26,88,74,1812,0.2281,0.26,0.2430,0.1814,114,0.26,0.26
Max Fusion,R2,2.4308,45,141,55,1759,0.2419,0.45,0.3147,0.1658,186,0.30,0.30


## **Step 16 — Save the R2-5 validation thresholds**

In [62]:
validation_records = []

for model_name, result in threshold_results.items():
    validation_records.append({
        "replication": REPLICATION,
        "model": model_name,
        **result
    })

for model_name, result in fusion_threshold_results.items():
    validation_records.append({
        "replication": REPLICATION,
        "model": model_name,
        **result
    })

validation_df = pd.DataFrame(validation_records)

validation_file = (
    output_dir / f"{REPLICATION}_validation.csv"
)

validation_df.to_csv(
    validation_file, index=False
)

print("Saved:", validation_file.resolve())

display(validation_df.round(4))

Saved: <PROJECT_ROOT>\results\robustness\R2_validation.csv


,replication,model,threshold,precision,recall,f1,fp
0,R2,Behavioral,0.5305,0.1419,0.44,0.2146,133
1,R2,Fund Flow Core,0.6217,0.4091,0.18,0.2500,13
2,R2,Combined Core,0.5397,0.2308,0.36,0.2812,60
3,R2,Fund Flow Extended,0.5744,0.4400,0.22,0.2933,14
4,R2,Mean Fusion,1.8234,0.2407,0.26,0.2500,41
5,R2,Max Fusion,2.4308,0.2381,0.40,0.2985,64
